# 08 · Monitoring

**Role of this layer.** Every stage of every run (and every replay) appends one
JSON object to `runs/events.jsonl`. The log is append-only, so it doubles as an
audit trail and could be shipped to a log platform as is.

The Monitoring page reads two sources, and the functions below (the ones behind
the page) turn them into numbers:
- the **run records**, one per run in its final state: activity, human review,
  tokens and cost, latency and failures;
- the **event log**, for what can happen several times in a run: guardrail
  firings, capability executions and replays.

To make sure there is something to show, this notebook first runs the five example
workflows through the graph in mock mode (approving only 1 and 5, as `cli demo` does).

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
from quantlab.graph import run_graph
from quantlab.mock_scenarios import SCENARIOS

for scenario in SCENARIOS:
    state = run_graph(
        scenario.request, list(scenario.dataset_ids), "mock", approve=scenario.auto_approve
    )
    print(scenario.id, state["status"])

chf_safe_haven_success completed
spread_frequency_mismatch_block blocked
arima_forecast_block blocked
prompt_injection_unknown_capability blocked


buy_the_dip_tech completed


In [3]:
from quantlab.lineage import load_runs
from quantlab.monitoring import activity, monitored_runs, read_events

records = load_runs(settings.runs_dir)
events = read_events(settings.events_path)
counted = monitored_runs(records)
print(len(records), "run records,", len(counted), "counted by the page,", len(events), "events")
pd.Series(activity(records))

372 run records, 105 counted by the page, 4295 events


total                      372.000000
completed                  145.000000
blocked                     82.000000
planner_refused             84.000000
failed                       0.000000
rejected                     4.000000
completed_share              0.389785
blocked_share                0.220430
planner_refused_share        0.225806
failed_share                 0.000000
rejected_share               0.010753
completed_percent           39.000000
blocked_percent             22.000000
planner_refused_percent     23.000000
failed_percent               0.000000
rejected_percent             1.000000
dtype: float64

The page counts only the decided runs started on the Analysis page
(`monitored_runs`, printed above), and replays from every source: these headless runs
are not among them. To show what each function computes, the cells below apply them
to every run record.

`activity` splits the runs into five outcomes that do not overlap (completed, blocked
by a deterministic guardrail, refused by the planner with P01, failed, rejected by the
reviewer), so they add up to the total; `_percent` holds the whole percentages the
page shows, which add up to 100.

In [4]:
from quantlab.monitoring import (
    completed_with_warnings,
    grounded_narratives,
    replay_results,
    review_outcomes,
)

pd.Series(
    {
        **review_outcomes(records),
        **replay_results(events),
        **grounded_narratives(records),
        **completed_with_warnings(records),
    }
)

approved               142.000000
rejected                 4.000000
approval_rate            0.972603
edited                  12.000000
edit_rate                0.084507
approval_median_s        0.041000
approval_p95_s          81.788400
replays                 17.000000
matching                17.000000
replay_match_rate        1.000000
narratives             145.000000
grounded               145.000000
grounded_rate            1.000000
completed              145.000000
with_warnings           58.000000
with_warnings_share      0.400000
dtype: float64

Every guardrail and every capability, zeros included, most frequent first:

In [5]:
from quantlab.monitoring import capability_counts, guardrail_counts

pd.DataFrame(guardrail_counts(events))

,code,name,action,count
0,V10,MATERIAL_ASSUMPTION,APPROVE,630
1,V09,LOOK_AHEAD,APPROVE,94
2,P01,PLANNER_REFUSED,BLOCK,84
3,E04,ALIGNMENT_LOSS,WARN,73
4,V08,FREQUENCY_MISMATCH,BLOCK,51
5,V03,INVALID_PARAM,BLOCK,42
6,V02,UNKNOWN_CAPABILITY,BLOCK,41
7,R01,REQUEST_TOO_LONG,BLOCK,6
8,V05,BUDGET_EXCEEDED,BLOCK,1
9,E05,LOW_EFFECTIVE_SAMPLE,WARN,1


In [6]:
pd.DataFrame(capability_counts(events))

,capability,code,count
0,load_series,C01,205
1,resample,C02,187
2,flag_regime,C07,140
3,forward_return,C08,114
4,hac_mean_difference,C10,114
5,compute_returns,C03,73
6,compare_groups,C09,54
7,rolling_volatility,C06,26
8,spread,C05,16
9,difference,C04,1


Cost per run, from the tokens the API reported and the prices in
`config.LLM_PRICES_PER_MILLION_TOKENS`. Mock runs call no LLM and are left out, so
in mock mode the table has no runs and the cache saved nothing:

In [7]:
from quantlab.config import LLM_PRICES_PER_MILLION_TOKENS
from quantlab.monitoring import cache_savings, cost_table

display(pd.DataFrame(cost_table(records, LLM_PRICES_PER_MILLION_TOKENS)))
pd.Series(cache_savings(records, LLM_PRICES_PER_MILLION_TOKENS))

,stage,runs,input_mean,input_p95,output_mean,output_p95,cost_mean,cost_p95,cost_total
0,planner,96,4323.593750,5361.75,485.854167,1015.50,0.000675,0.001044,0.064827
1,narrator,96,431.458333,1578.00,52.385417,178.25,0.000069,0.000237,0.006657
2,total,96,4755.052083,6824.00,538.239583,1140.00,0.000745,0.001154,0.071484


planner_calls             92.000000
cache_hits                10.000000
hit_rate                   0.108696
saved_input_tokens     47946.000000
saved_output_tokens     5679.000000
saved_cost                 0.007634
dtype: float64

Latency per stage (mean and 95th percentile, milliseconds), and the most recent
failures with the stage and the guardrail that stopped them:

In [8]:
from quantlab.monitoring import latency_table, recent_failures

pd.DataFrame(latency_table(records))

,stage,runs,mean_ms,p95_ms,recorded
0,Request check,0,NaN,NaN,False
1,Planner,366,1681.542077,8703.225,True
2,Validator,366,0.155601,0.400,True
3,Executor,145,152.876345,276.702,True
4,Narrator,145,689.831724,3526.360,True


In [9]:
pd.DataFrame(recent_failures(records, limit=5))

,time,run_id,status,stage,code,message
0,2026-10-20T14:44:00.121Z,20261020-144400-bc8a09,blocked,Validator,V02,The plan was rejected (V02): write a new request.
1,2026-10-20T14:44:00.061Z,20261020-144400-8a212a,blocked,Planner,P01,The plan has blocking issues: P01:plan
2,2026-10-20T14:43:59.997Z,20261020-144359-c337cd,blocked,Validator,V08,The plan has blocking issues: V08:s4
3,2026-10-20T14:42:56.373Z,20261020-144256-363aba,blocked,Validator,V02,The plan was rejected (V02): write a new request.
4,2026-10-20T14:42:56.329Z,20261020-144256-f9ce3d,blocked,Planner,P01,The plan has blocking issues: P01:plan
